# One SAP → AE TFL Shell | End-to-End MVP

**Input:** Read `SAP_001_00.pdf` automatically from the established Google Drive folder: `MyDrive/Colab Notebooks/agent/midterm demo/`. **Output:** save the editable DOCX and JSON audit reports back to a subfolder alongside that SAP, and save a ZIP in the same Google Drive directory.

This notebook is a *thin GitHub package client*: Git clone → mount Drive and locate the SAP → one pipeline call → save/export results. **No manual SAP upload, intermediate ZIP, LLM API key or Adaptation Plan approval.**

**Pilot scope:** Overall AE Table (optimized for RP6530-2101). Output is a structural draft, not a clinically validated TFL Shell.

## Step 0 — Clone and install the official GitHub Agent package

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'feature/official-library-colab'
PROJECT_ROOT = Path('/content/tfl-automatic-ae-pilot')
if (PROJECT_ROOT / '.git').is_dir():
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', '-B', BRANCH, 'FETCH_HEAD'])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError('PROJECT_ROOT exists but is not a Git repository')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)])
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('GitHub package:', subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'], text=True).strip())

## Step 1 — Mount Google Drive and locate the existing SAP

Same Drive path as Colab 04/06. **No file picker or manual upload.** You may need to authorize the Google Drive mount once per Colab runtime.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_SAP_RELATIVE_PATH = "Colab Notebooks/agent/midterm demo/SAP_001_00.pdf"
SAP_PATH = Path("/content/drive/MyDrive") / DRIVE_SAP_RELATIVE_PATH
DRIVE_SAP_DIRECTORY = SAP_PATH.parent

if not SAP_PATH.is_file():
    existing_pdfs = sorted(
        x.name for x in DRIVE_SAP_DIRECTORY.glob("*.pdf")
    ) if DRIVE_SAP_DIRECTORY.is_dir() else []
    raise FileNotFoundError(
        f"SAP not found: {SAP_PATH}\n"
        f"Drive folder exists: {DRIVE_SAP_DIRECTORY.is_dir()}\n"
        f"Available PDFs: {existing_pdfs}\n"
        "Check DRIVE_SAP_RELATIVE_PATH; no manual upload is needed."
    )

print("Google Drive SAP:", SAP_PATH)
print("File size (MB):", round(SAP_PATH.stat().st_size / (1024 * 1024), 2))

## Step 2 — Run the complete Agent pipeline (one call)

All extraction and structural DOCX generation are **GitHub package implementations**, not notebook cells:

SAP parsing → AE specification → official keyword template retrieval → diagnostic adaptation plan → structural Word rebuild → validation. No manual adaptation-plan approval is needed to generate the **unapproved** draft.

In [ ]:
from tfl_agent.agent.end_to_end_ae_pilot import run_single_sap_ae_pilot
OUTPUT_DIR = DRIVE_SAP_DIRECTORY / "AE_TFL_End_to_End_Outputs"
result = run_single_sap_ae_pilot(
    sap_path=SAP_PATH,
    library_path=PROJECT_ROOT / 'library',
    output_dir=OUTPUT_DIR,
)
print(json.dumps({
    'Status': result['status'],
    'Target': result['target_id'],
    'Official Library count': result['approved_library_count'],
    'Selected style template': result['chosen_style_template_id'],
    'Target AE rows': result['generation']['row_count'],
    'Word columns': result['generation']['column_count'],
    'Structural validation': result['validation']['valid_structural_shell'],
    'Clinical approval': result['validation']['clinical_validity_verified'],
}, indent=2))
assert result['validation']['valid_structural_shell'], result['validation']['structural_errors']

## Step 3 — Save outputs in the same Google Drive directory

The DOCX and audit JSONs are already written under `MyDrive/Colab Notebooks/agent/midterm demo/AE_TFL_End_to_End_Outputs/`. This cell also creates `AE_TFL_End_to_End_Outputs.zip` in `midterm demo/` for convenient sharing. No additional upload or download dialog.

In [ ]:
from IPython.display import display, FileLink

assert OUTPUT_DIR.is_dir(), "Run Step 2 first."
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)
    if path.suffix.lower() == ".docx":
        display(FileLink(str(path), result_html_prefix="Editable AE structural draft: "))

ARCHIVE_PATH = DRIVE_SAP_DIRECTORY / "AE_TFL_End_to_End_Outputs.zip"
shutil.make_archive(
    str(ARCHIVE_PATH.with_suffix("")), "zip", root_dir=OUTPUT_DIR
)
assert ARCHIVE_PATH.is_file(), "ZIP was not created"
print("Output folder:", OUTPUT_DIR)
print("ZIP saved on Google Drive:", ARCHIVE_PATH)
print("ZIP size (MB):", round(ARCHIVE_PATH.stat().st_size / (1024 * 1024), 2))

**MVP limits:** Works only for a recognizable Overall AE SAP section and an explicit two-group design. No patient data are estimated; numeric cells remain `xx`. Footnote applicability, denominator and formatting need clinical review. The result is a structural shell **draft**, not an approved clinical TFL.